In [ ]:
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import pandas as pd
from collections import defaultdict
import sklearn
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans
from scipy.stats import percentileofscore
from scipy.sparse.csgraph import minimum_spanning_tree, shortest_path

In [ ]:
def get_amyloid_dict(amyloid_df):
    amyloid_df['AMYLOID'] = amyloid_df.groupby('DXGrp')['AMYLOID'].transform(
        lambda x: x.fillna(x.mean())
    )
    amyloid_dict = {}
    for _, row in amyloid_df.iterrows():
        key = (str(row['RID']), round(row['AGE'], 2))
        amyloid_dict[key] = row['AMYLOID']

    return amyloid_dict


In [ ]:
train_amyloid_dict = get_amyloid_dict(
    pd.read_csv('./data/ADNI1GO234/splits/trial2/preadj_train.csv')
)

amyloid_threshold = 10  # value

values = list(train_amyloid_dict.values())
# print(f"Centiloid value at {amyloid_threshold*100} percentile:", np.percentile(values, amyloid_threshold*100))
print(f"Amyloid value is 10 at {percentileofscore(values, 10, kind='rank')/100.0} percentile")

In [ ]:
# Load data functions
def process_data(trial, dataset="train", impute_lb=False):
    results_path = './results/ADNI1GO234/LSP/'
    time_label = next(folder for folder in os.listdir(results_path) if folder.startswith(f"trial{trial}_"))
    data_path = f"{results_path}{time_label}/{dataset}_results.npy"
    dataset_path = f"./data/ADNI1GO234/splits/trial{trial}/preadj_{dataset}.csv"
    dataset_df = pd.read_csv(dataset_path)
    
    print(f"Loading data from {data_path}")

    data = np.load(data_path, allow_pickle=True).item()
    if impute_lb:
        data = fill_missing_lb(data)

    az = data['pc'] # pca coordinates
    aRID = data['RID']
    alb = data['lb']
    aage = np.round(data['age'], 2)
    
    unique_subjects = {}
    for z, rid, lb, age in zip(az, aRID, alb, aage):
        if rid not in unique_subjects:
            unique_subjects[rid] = {
                'z': z,
                'age': age,
                'lb': [lb]
            }
        else:
            if age > unique_subjects[rid]['age']:
                unique_subjects[rid]['z'] = z
                unique_subjects[rid]['age'] = age
            if lb not in unique_subjects[rid]['lb']:
                unique_subjects[rid]['lb'].append(lb)

    uz = np.array([info['z'] for info in unique_subjects.values()])
    urid = np.array(list(unique_subjects.keys()))
    ulb = [info['lb'] for info in unique_subjects.values()]
    uage = np.array([info['age'] for info in unique_subjects.values()])
    
    # ----- GET AMYLOID DICT -----
    amyloid_dict = get_amyloid_dict(dataset_df)

    return az, aRID, alb, aage, uz, urid, ulb, uage, amyloid_dict, dataset_df

def fill_missing_lb(data):
    # Extract columns
    RID = data["RID"]
    age = data["age"]
    lb = data["lb"].copy()

    filled_lb = lb.copy()

    for rid in np.unique(RID):
        # Select indices for this RID
        idx = np.where(RID == rid)[0]

        # Sort by age
        sorted_idx = idx[np.argsort(age[idx])]
        lbs = lb[sorted_idx]

        for i in range(len(lbs)):
            if lbs[i] == -1:
                val = None
                # Check previous
                for j in range(i - 1, -1, -1):
                    if lbs[j] != -1:
                        val = lbs[j]
                        break
                # Check next
                if val is None:
                    for j in range(i + 1, len(lbs)):
                        if lbs[j] != -1:
                            val = lbs[j]
                            break
                if val is not None:
                    lbs[i] = val
        
        # assign filled values back to the right indices
        filled_lb[sorted_idx] = lbs

    # Return a new dict with filled lb
    new_data = data.copy()
    new_data["lb"] = filled_lb
    return new_data

In [ ]:
# learn_graph_2d functions

def kmeans_and_sort(X, n_clusters):
    """Cluster with KMeans and sort the centers by coordinate."""
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    cluster_centers = kmeans.cluster_centers_
    
    # Sort by coordinate.
    sorted_indices = np.lexsort((cluster_centers[:, 1], cluster_centers[:, 0]))
    cluster_centers = cluster_centers[sorted_indices]
    
    new_labels = np.zeros_like(labels)
    for new_id, old_id in enumerate(sorted_indices):
        new_labels[labels == old_id] = new_id
    
    return new_labels, cluster_centers

def detect_branchs(branch_degree, cluster_centers):
    cluster_centers_graph = kneighbors_graph(cluster_centers, 3, mode='distance', include_self=False)
    mst = minimum_spanning_tree(cluster_centers_graph)
    
    G = nx.Graph()      # Convert to NetworkX graph
    mst_coo = mst.tocoo()
    for i, j, weight in zip(mst_coo.row, mst_coo.col, mst_coo.data):
        G.add_edge(i, j, weight=weight)
    
    # Select trajectory starting point (nodes with degree 1 in MST)
    leaf_nodes = [node for node in G.nodes if G.degree(node) == 1]
    root = leaf_nodes[0] if leaf_nodes else 0
    
    # Calculate pseudotime for all cluster centers (shortest path to root)
    lengths = shortest_path(mst, directed=False, indices=root)
    pseudotime = lengths / np.max(lengths) if np.max(lengths) > 0 else lengths
    
    # Identify branch points (nodes with degree >= branch_degree in MST)
    branch_points = [node for node in G.nodes if G.degree(node) >= branch_degree]
    
    # Construct directed tree with root as starting point (BFS tree)
    T = nx.bfs_tree(G, source=root)
    
    # Analyze branches
    branches = []
    for bp in branch_points:
        if bp not in T:
            continue
        for child in T.successors(bp):
            branch = [bp]
            current = child
            while (current not in branch_points) and (T.out_degree(current) == 1):
                branch.append(current)
                next_nodes = list(T.successors(current))
                if len(next_nodes) == 0:
                    break
                current = next_nodes[0]
            branch.append(current)
            branches.append(branch)
    
    return branches, branch_points, G, root, pseudotime

def visualize_trajectory(X, labels, cluster_centers, G, root, branch_points, normal_branch, AD_branch, pseudotime):
    plt.figure(figsize=(10, 4))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=10, cmap='tab20', edgecolors='k', alpha=0.5)
    center_scatter = plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], c=pseudotime, s=100, cmap='viridis_r', edgecolors='black')
    for i, (x, y) in enumerate(cluster_centers):
        plt.text(x, y, str(i), fontsize=10, ha='center', va='center', color='white', fontweight='bold')
    for i, j in G.edges():
        plt.plot([cluster_centers[i, 0], cluster_centers[j, 0]], [cluster_centers[i, 1], cluster_centers[j, 1]], 'gray', linewidth=2)
    plt.scatter(cluster_centers[root, 0], cluster_centers[root, 1], color='red', s=120)
    for bp in branch_points:
        plt.scatter(cluster_centers[bp, 0], cluster_centers[bp, 1], color='orange', s=120)
    plt.plot(cluster_centers[normal_branch][:, 0], cluster_centers[normal_branch][:, 1], '-', color="b", linewidth=3, label="Normal Branch")
    plt.plot(cluster_centers[AD_branch][:, 0], cluster_centers[AD_branch][:, 1], '-', color="r", linewidth=3, label="AD Branch")
    plt.legend(loc='upper right')
    # plt.colorbar(center_scatter, label="Pseudotime")
    plt.show()
        
def learn_graph_2d(X, n_range=[10,32], branch_degree=3):
    for n_clusters in range(n_range[1], n_range[0], -1):
        new_labels, cluster_centers = kmeans_and_sort(X, n_clusters)
        cluster_points = {i: X[new_labels == i] for i in range(n_clusters)}
        
        # Compute branches.
        branches, branch_points, G, root, pseudotime = detect_branchs(branch_degree, cluster_centers)
        
        if len(branches) == 2:
            break
    
    if len(branches) != 2:
        print(f"Found {len(branches)} branches")
        return None
    
    if branches[0][-1] > branches[1][-1]:
        AD_branch = branches[0]
        normal_branch = branches[1]
    else:
        AD_branch = branches[1]
        normal_branch = branches[0]
    
    visualize_trajectory(X, new_labels, cluster_centers, G, root, branch_points, normal_branch, AD_branch, pseudotime)
    
    # Return the points in each cluster and the cluster centers along each branch segment.
    return cluster_points, normal_branch, AD_branch

In [ ]:
# cal_pseudotime functions

import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors, KDTree
from scipy.interpolate import splprep, splev

def compute_spline_pseudotime(points, ref_vector=None, k=10, smooth=0.5, n_eval=400, saved_tck=None):
    points = np.asarray(points)
    
    # --- Step 1: Fit or Load Spline ---
    if saved_tck is not None:
        # Use existing model
        tck = saved_tck
    else:
        # Fit new model
        N = len(points)
        # 1) Build kNN graph to get approximate ordering
        nbrs = NearestNeighbors(n_neighbors=min(k + 1, N)).fit(points)
        dists, idxs = nbrs.kneighbors(points)

        G = nx.Graph()
        for i in range(N):
            for j, dist in zip(idxs[i, 1:], dists[i, 1:]):  # skip self
                G.add_edge(i, int(j), weight=float(dist))

        # 2) Find approximate endpoints using graph diameter
        try:
            lengths0 = nx.single_source_dijkstra_path_length(G, 0, weight='weight')
            a = max(lengths0, key=lengths0.get)
            lengthsa = nx.single_source_dijkstra_path_length(G, a, weight='weight')
            b = max(lengthsa, key=lengthsa.get)
            
            # 3) Use shortest path from a to b as initial ordering
            path = nx.shortest_path(G, a, b, weight='weight')
            P = points[path]
        except:
            # Fallback for very small/disconnected datasets
            # Sort by first principal component or simple x-axis
            idx_sort = np.argsort(points[:, 0])
            P = points[idx_sort]

        # 4) Fit smooth spline curve along this path
        # s is the smoothing factor
        try:
            (tck, u) = splprep(P.T, s=smooth, k=min(3, len(P)-1)) 
        except:
            # Fallback if too few points for cubic spline, try linear (k=1)
            (tck, u) = splprep(P.T, s=smooth, k=1)

    # --- Step 2: Project Points onto Spline ---
    
    # Generate dense points along the spline to act as "magnets"
    us = np.linspace(0, 1, n_eval)
    xs, ys = splev(us, tck)

    # Compute cumulative arc length for the dense curve
    S = np.cumsum(np.hypot(np.diff(xs), np.diff(ys)))
    S = np.concatenate([[0.0], S])
    S = (S - S.min()) / (S.max() - S.min() + 1e-12) # Normalize 0 to 1

    # Project each actual data point to the nearest sample on the dense curve
    XYs = np.vstack([xs, ys]).T
    tree = KDTree(XYs)
    dist, idx = tree.query(points, k=1)
    pseudotime = S[idx[:, 0]]

    # --- Step 3: Direction Correction ---
    # Optional: ensure pseudotime correlates positively with a reference (e.g., Time or X-axis)
    if ref_vector is not None:
        corr = np.corrcoef(pseudotime, ref_vector)[0, 1]
        if corr < 0:
            pseudotime = 1 - pseudotime

    return pseudotime, tck


def cal_pseudotime(all_points, normal_points, AD_points, pseudotime_model=None):
    # Filter common points (start points shared by both)
    common_points = np.array([
        point for point in all_points 
        if not any(np.array_equal(point, p) for p in normal_points) 
        and not any(np.array_equal(point, p) for p in AD_points)
    ])
    
    normal_curve_points = np.vstack([common_points, normal_points]) if len(common_points) > 0 else normal_points
    AD_curve_points = np.vstack([common_points, AD_points]) if len(common_points) > 0 else AD_points
    
    # Convert to DataFrame for easier handling
    df_normal = pd.DataFrame({
        'x': [p[0] for p in normal_curve_points],
        'y': [p[1] for p in normal_curve_points],
        'group': 'normal'
    })
    df_AD = pd.DataFrame({
        'x': [p[0] for p in AD_curve_points],
        'y': [p[1] for p in AD_curve_points],
        'group': 'AD'
    })
    
    # --- Calculate Pseudotime (Direct Spline Usage) ---
    
    tck_normal = None
    tck_ad = None
    
    if pseudotime_model is not None:
        # Load existing spline models if available
        tck_normal = pseudotime_model.get("tck_normal")
        tck_ad = pseudotime_model.get("tck_ad")
        

    # Compute for Normal
    pt_normal, final_tck_normal = compute_spline_pseudotime(
        df_normal[['x', 'y']].values,
        ref_vector=df_normal['x'].values,
        smooth=0.5,
        saved_tck=tck_normal
    )
    df_normal['fitted_pseudotime'] = pt_normal

    # Compute for AD
    pt_ad, final_tck_ad = compute_spline_pseudotime(
        df_AD[['x', 'y']].values,
        ref_vector=df_AD['x'].values,
        smooth=0.5,
        saved_tck=tck_ad
    )
    df_AD['fitted_pseudotime'] = pt_ad

    # --- Normalization and Merging ---
    # If we are predicting, we should arguably use saved bounds, but 0-1 is standard for pseudotime.
    if pseudotime_model is not None:
        global_min = pseudotime_model.get("vmin")
        global_max = pseudotime_model.get("vmax")
        normal_min = pseudotime_model.get("normal_min")
        normal_max = pseudotime_model.get("normal_max")
        ad_min = pseudotime_model.get("ad_min")
        ad_max = pseudotime_model.get("ad_max")
    else:
        global_min = min(df_normal['fitted_pseudotime'].min(), df_AD['fitted_pseudotime'].min())
        global_max = max(df_normal['fitted_pseudotime'].max(), df_AD['fitted_pseudotime'].max())
        normal_min = df_normal['fitted_pseudotime'].min()
        normal_max = df_normal['fitted_pseudotime'].max()
        ad_min = df_AD['fitted_pseudotime'].min()
        ad_max = df_AD['fitted_pseudotime'].max()
    
    # Normalize to global 0-1 range (handles slight variations between the two curves)
    df_normal_norm = (df_normal['fitted_pseudotime'] - normal_min) / (normal_max - normal_min + 1e-12)
    df_AD_norm = (df_AD['fitted_pseudotime'] - ad_min) / (ad_max - ad_min + 1e-12)
    
    df_normal['fitted_pseudotime'] = df_normal_norm
    df_AD['fitted_pseudotime'] = df_AD_norm
    
    # Combine results
    df_final = pd.concat([df_normal, df_AD], ignore_index=True)
    
    # Map fitted pseudotime back to all_points (using coordinates as keys)
    pseudotime_mapping = {}
    for _, row in df_final.iterrows():
        # Rounding keys slightly can help with floating point mismatch, 
        # but using exact tuple usually works if points came from same source
        pseudotime_mapping[(row['x'], row['y'])] = row['fitted_pseudotime']
    
    pseudotime_all_points = np.array([
        pseudotime_mapping.get((point[0], point[1]), np.nan) 
        for point in all_points
    ])
        
    ############
    # Plotting #
    ############
    plt.figure(figsize=(15, 4))
    
    plt.subplot(1, 3, 1)
    plt.title("Normal Curve")
    plt.scatter(all_points[:, 0], all_points[:, 1], color='grey', alpha=0.3, label='All Points')
    scatter_normal = plt.scatter(df_normal['x'], df_normal['y'], c=df_normal_norm, cmap='Blues', vmin=0, vmax=1, label='Normal Curve')
    plt.colorbar(scatter_normal, label='Pseudotime')
    # --- NEW: Draw the fitted curve ---
    u_curve = np.linspace(0, 1, 1024)
    x_curve, y_curve = splev(u_curve, final_tck_normal)
    plt.plot(x_curve, y_curve, color='red', linewidth=2, label='Fitted Spline')
    
    plt.subplot(1, 3, 2)
    plt.title("Pseudotime for AD Points")
    plt.scatter(all_points[:, 0], all_points[:, 1], color='grey', alpha=0.3, label='All Points')
    scatter_AD = plt.scatter(df_AD['x'], df_AD['y'], c=df_AD_norm, cmap='Reds', vmin=0, vmax=1, label='AD Curve')
    plt.colorbar(scatter_AD, label='Pseudotime')
    # --- NEW: Draw the fitted curve ---
    u_curve = np.linspace(0, 1, 1024)
    x_curve, y_curve = splev(u_curve, final_tck_ad)
    plt.plot(x_curve, y_curve, color='red', linewidth=2, label='Fitted Spline')
    
    plt.subplot(1, 3, 3)
    plt.title("Pseudotime for All Points")
    # Handle NaNs in plotting if any points failed mapping
    valid_mask = ~np.isnan(pseudotime_all_points)
    scatter_all = plt.scatter(
        all_points[valid_mask, 0], 
        all_points[valid_mask, 1], 
        c=pseudotime_all_points[valid_mask], 
        cmap='Greens', 
        vmin=0, vmax=1
    )
    plt.colorbar(scatter_all, label='Pseudotime')
    
    plt.tight_layout()
    plt.show()
    
    # Save the spline parameters (tck) instead of LME objects
    new_pseudotime_model = {
        "tck_normal": final_tck_normal,
        "tck_ad": final_tck_ad,
        "vmin": global_min,
        "vmax": global_max,
        "normal_min": normal_min,
        "normal_max": normal_max,
        "ad_min": ad_min,
        "ad_max": ad_max
    }
    
    return pseudotime_all_points, df_normal_norm, df_AD_norm, new_pseudotime_model

In [ ]:
# cal_violation functions

def cal_violation(az, aRID, aage, apseudotime, thresholds=[0]):
    subj_dict = {}
    for i in range(len(az)):
        if aRID[i] not in subj_dict:
            subj_dict[aRID[i]] = {
                "z": [],
                "age": [],
                "pseudotime": []
            }
        subj_dict[aRID[i]]["z"].append(az[i])
        subj_dict[aRID[i]]["age"].append(aage[i])
        subj_dict[aRID[i]]["pseudotime"].append(apseudotime[i])
    # sort by age
    for key in subj_dict:
        sorted_indices = np.argsort(subj_dict[key]["age"])
        subj_dict[key]["z"] = np.array(subj_dict[key]["z"])[sorted_indices]
        subj_dict[key]["age"] = np.array(subj_dict[key]["age"])[sorted_indices]
        subj_dict[key]["pseudotime"] = np.array(subj_dict[key]["pseudotime"])[sorted_indices]
        
    # calculate violation for multiple thresholds
    thresholds = np.array(thresholds)
    violation_rates_subj = []
    violation_rates_pair = []
    avg_violation_gaps = []

    for threshold in thresholds:
        violation_subj = 0
        violation_pair = 0
        total_subjs = len(subj_dict)
        total_pairs = 0
        violation_gaps = []

        for key in subj_dict:
            violation_flag = 0
            for i in range(len(subj_dict[key]["z"]) - 1):
                total_pairs += 1
                gap = subj_dict[key]["pseudotime"][i + 1] - subj_dict[key]["pseudotime"][i]
                if gap < -threshold:
                    violation_flag = 1
                    violation_pair += 1
                    violation_gaps.append(abs(gap))
            violation_subj += violation_flag

        violation_rate_subj = violation_subj / total_subjs
        violation_rate_pair = violation_pair / total_pairs
        avg_violation_gap = np.mean(violation_gaps) if violation_gaps else 0

        violation_rates_subj.append(violation_rate_subj)
        violation_rates_pair.append(violation_rate_pair)
        avg_violation_gaps.append(avg_violation_gap)

    # Plot the results
    plt.figure(figsize=(7, 3), dpi=100)

    # Violation ratio plot
    plt.subplot(1, 2, 1)
    # plt.plot(thresholds, violation_rates_subj, label="Violation Ratio (Subjects)", marker='o')
    plt.plot(thresholds, violation_rates_pair, label="Violation Ratio (Pairs)", marker='x')
    plt.grid(True)
    plt.xlabel("Threshold")
    plt.ylabel("Violation Ratio")
    # plt.title("Violation Ratio vs Threshold")
    # plt.legend()

    # Violation gap plot
    plt.subplot(1, 2, 2)
    plt.plot(thresholds, avg_violation_gaps, label="Average Violation Gap", marker='s', color='r')
    plt.grid(True)
    plt.xlabel("Threshold")
    plt.ylabel("Average Violation Gap")
    # plt.title("Violation Gap vs Threshold")
    # plt.legend()

    plt.tight_layout()
    plt.show()
    

In [ ]:
# cal_accuracy functions

def cal_accuracy(RID, Z, LB, AGE, common_points, normal_points, AD_points, amyloid_dict, threshold=amyloid_threshold):
    # Define categories based on transitions
    categories = [f"{i}->{j}" for i in range(1, 5) for j in range(i, 5)]
    
    # Initialize dictionaries for total subjects and correct predictions per branch group
    groups = ['AD_branch', 'normal_branch', 'total']
    total_counts = {g: {cat: 0 for cat in categories} for g in groups}
    correct_counts = {g: {cat: 0 for cat in categories} for g in groups}
    overall_total = {g: 0 for g in groups}
    overall_correct = {g: 0 for g in groups}
    
    branch_amyloid = {'common_branch': [], 'normal_branch': [], 'AD_branch': []}
    
    for i in range(len(Z)):
        rid = RID[i]
        z = Z[i]
        lb1 = LB[i][0]
        lb2 = LB[i][-1]  
        age = AGE[i]
        cat = f"{lb1}->{lb2}"
        # Check for amyloid data and add to appropriate branch collection
        try:
            amyloid_value = amyloid_dict[(rid, age)]
            
            if z in normal_points:
                branch_amyloid['normal_branch'].append(amyloid_value)
            elif z in AD_points:
                branch_amyloid['AD_branch'].append(amyloid_value)
            else:
                branch_amyloid['common_branch'].append(amyloid_value)
        except KeyError:
            print(f"({rid}, {age}) not found in amyloid data.")
            continue  # Skip if no amyloid data available
        
        # Determine the expected branch based on clinical data
        if -1 in [lb1, lb2] or lb2 < lb1:
            continue
        elif z in common_points:
            continue
        elif lb2 in [3, 4]:
            expected = 'AD_branch'
            is_correct = (z in AD_points)
        else:
            if amyloid_dict[(rid, age)] > threshold and lb2 == 1:
                continue
            expected = 'normal_branch'
            is_correct = (z in normal_points) or (z in common_points)

        # Update overall total and correct counts
        overall_total['total'] += 1
        if is_correct:
            overall_correct['total'] += 1
            correct_counts['total'][cat] += 1
        total_counts['total'][cat] += 1
        
        # Update group-specific counts
        overall_total[expected] += 1
        total_counts[expected][cat] += 1
        if is_correct:
            overall_correct[expected] += 1
            correct_counts[expected][cat] += 1

    # Define a helper to compute accuracy percentage
    def _calc_accuracy(correct, total):
        return f"{100*correct/total:6.2f}%" if total > 0 else "  N/A "

    # Print category-wise accuracy matrix
    print("\nCategory-wise accuracy matrix:")
    print("     ", end="")
    for j in range(1, 5):
        print(f"{   j   :17}", end="")
    print()

    for i in range(1, 5):
        print(f"{i}    ", end="")
        for j in range(1, 5):
            if j < i:
                print(f"{' ':17}", end="")
            elif f"{i}->{j}" in categories:
                acc = _calc_accuracy(correct_counts['total'][f"{i}->{j}"], total_counts['total'][f"{i}->{j}"])
                count = f"({correct_counts['total'][f'{i}->{j}']}/{total_counts['total'][f'{i}->{j}']})"
                print(f"{acc:7}{count:10}", end="")
            print("    ", end="")
        print()
        
    ####################
    # Confusion matrix #
    ####################
    # (AD: 1->4, 2->4, 3->4, 4->4; NC: 1->1)
    print("\nConfusion matrix:")
    print("     ", end="")
    for j in ["AD_branch", "normal_branch"]:
        print(f"{j:9}", end="")
    print()
    TP_count = correct_counts['total']['1->4'] + correct_counts['total']['2->4'] + correct_counts['total']['3->4'] + correct_counts['total']['4->4']
    TN_count = correct_counts['total']['1->1']
    FN_count = total_counts['total']['1->4'] + total_counts['total']['2->4'] + total_counts['total']['3->4'] + total_counts['total']['4->4'] - TP_count
    FP_count = total_counts['total']['1->1'] - TN_count
    print(f"AD  {TP_count:8} {FN_count:8}")
    print(f"NC  {FP_count:8} {TN_count:8}")
    accuracy_confuse = (TP_count + TN_count) / (TP_count + TN_count + FP_count + FN_count)
    precision_confuse = TP_count / (TP_count + FP_count)
    recall_confuse = TP_count / (TP_count + FN_count)
    f1_confuse = 2 * (precision_confuse * recall_confuse) / (precision_confuse + recall_confuse)
    specificity_confusion = TN_count / (TN_count + FP_count)
    print(f'{"Accuracy":^12}{"F1-score":^12}{"Precision":^12}{"Recall":^12}{"Specificity":^12}')
    print(f"{accuracy_confuse:^12.4f}{f1_confuse:^12.4f}{precision_confuse:^12.4f}{recall_confuse:^12.4f}{specificity_confusion:^12.4f}")

    ###################################
    # Amyloid Distribution Statistics #
    ###################################
    # print("\nAmyloid Distribution based on subjects:")
    # print("          \tMean\tStd\tMin\tMax\tN")
    # for branch in ['common_branch', 'normal_branch', 'AD_branch']:
    #     amyloid_values = branch_amyloid[branch]
    #     if amyloid_values:
    #         print(f"{branch}: \t{np.mean(amyloid_values):.3f}\t{np.std(amyloid_values):.3f}\t{np.min(amyloid_values):.3f}\t{np.max(amyloid_values):.3f}\t{len(amyloid_values)}")

    ###############################################
    # Plot amyloid distribution based on subjects #
    ###############################################
    # plt.figure(figsize=(8, 4))
    # branch_colors = {'common_branch': 'green', 'normal_branch': 'blue', 'AD_branch': 'red'}
    # for branch in ['common_branch', 'normal_branch', 'AD_branch']:
    #     amyloid_values = branch_amyloid[branch]
    #     if amyloid_values:
    #         plt.hist(amyloid_values, bins=20, alpha=0.5, label=branch, color=branch_colors[branch])
    # plt.axvline(x=threshold, color='r', linestyle='--', label='Threshold')
    # plt.xlabel('Amyloid SUVR')
    # plt.ylabel('Count')
    # plt.title('Distribution of Amyloid Values by Branch (Subjects)')
    # plt.legend()
    # plt.show()
        
    return {
        'AD_branch': _calc_accuracy(overall_correct['AD_branch'], overall_total['AD_branch']),
        'normal_branch': _calc_accuracy(overall_correct['normal_branch'], overall_total['normal_branch']),
        'total': _calc_accuracy(overall_correct['total'], overall_total['total']),
    }

In [ ]:
# Visualize functions

def draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, labeltoshow):
    if labeltoshow in [1]:
        selected_points = []
        for i in range(len(uRID)):
            rid = uRID[i]
            age = uage[i]
            if ulb[i] == [labeltoshow] and amyloid_dict[(rid, age)] <= amyloid_threshold:
                selected_points.append(uz[i])
        selected_points = np.array(selected_points)
    else:
        # Extract the selected points based on the label
        selected_points = [uz[i] for i in range(len(ulb)) if labeltoshow in ulb[i]]
        selected_points = np.array(selected_points)
                
    
    # Define a simple color map for labels
    label_color_map = {
        1: 'green',
        2: 'steelblue',
        3: 'brown',
        4: 'purple'
    }
    labels = {
        1: 'CN',
        2: 'EMCI',
        3: 'LMCI',
        4: 'AD'
    }

    # plt.figure(figsize=(3, 3))
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.scatter(az[:, 0], az[:, 1], c="grey", s=10, alpha=0.5, label='All data')

    # Pick the color for this label
    color = label_color_map.get(labeltoshow, "black")

    print(f"Number of points with label {labeltoshow}: {len(selected_points)}")
    plt.scatter(selected_points[:, 0], selected_points[:, 1], c=color, s=20, label=f"{labels[labeltoshow]}")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.scatter(uz[:, 0], uz[:, 1], c="grey", s=10, alpha=0.5)
    plt.scatter(selected_points[:, 0], selected_points[:, 1], c=color, s=20, label=f"{labels[labeltoshow]}")
    plt.legend()
    plt.show()

---

In [ ]:
for trial in [2]:  # [2, 30, 35]:
    for dataset in ["train"]:  # , "test"]:
        print(f"trial {trial} ({dataset}):")
        az, aRID, alb, aage, uz, uRID, ulb, uage, amyloid_dict, dataset_df = process_data(trial, dataset, impute_lb=True)
        try:
            cluster_points, normal_branch, AD_branch = learn_graph_2d(az, branch_degree=3)
            normal_branch = [int(x) for x in normal_branch]
            AD_branch = [int(x) for x in AD_branch]
        except Exception as e:
            print(e)
            continue
        normal_points = np.concatenate([cluster_points[i] for i in normal_branch])
        AD_points = np.concatenate([cluster_points[i] for i in AD_branch[1:]])
        print("Normal_branch:", normal_points.shape, normal_branch)
        print("AD_branch:    ", AD_points.shape, AD_branch[1:])

        draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, 1)
        draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, 2)
        # draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, 3)
        draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, 4)

# Control Branch

In [ ]:
def initial_trajectory(X, n_range=[10,32], branch_degree=3):
    for n_clusters in range(n_range[1], n_range[0], -1):
        new_labels, cluster_centers = kmeans_and_sort(X, n_clusters)
        cluster_points = {i: X[new_labels == i] for i in range(n_clusters)}
        
        # Compute branches.
        branches, branch_points, G, root, pseudotime = detect_branchs(branch_degree, cluster_centers)
        
        if len(branches) == 2:
            break
    
    if len(branches) != 2:
        print(f"Found {len(branches)} branches")
        return None
    
    if branches[0][-1] > branches[1][-1]:
        AD_branch = branches[0]
        normal_branch = branches[1]
    else:
        AD_branch = branches[1]
        normal_branch = branches[0]
        
    visualize_dict = {
        "X": X,
        "new_labels": new_labels,
        "cluster_centers": cluster_centers,
        "G": G,
        "root": root,
        "branch_points": branch_points,
        "normal_branch": normal_branch,
        "AD_branch": AD_branch,
        "pseudotime": pseudotime
    }
    
    return cluster_points, normal_branch, AD_branch, visualize_dict

def match_branch_ptime(az, branch_curve_z, apseudotime, branch_ptime):
    # Initialize ptime_branch with -1 values matching the shape of az.
    ptime_branch = np.full_like(apseudotime, -1)

    # Match each point in az to its index in branch_curve_z.
    for i, point in enumerate(az):
        for j, ref_point in enumerate(branch_curve_z):
            if np.array_equal(point, ref_point):
                ptime_branch[i] = branch_ptime[j]
                break
    return ptime_branch

def cluster_new_points(cluster_points, new_points):
    cluster_centers = {}
    for cluster_id, points in cluster_points.items():
        center = np.mean(points, axis=0)
        cluster_centers[cluster_id] = center
    
    cluster_points_single = {x: [] for x in cluster_points.keys()}
    new_clusters = []
    for point in new_points:
        min_distance = float('inf')
        closest_cluster = None
        
        for cluster_id, center in cluster_centers.items():
            distance = np.linalg.norm(point - center)
            if distance < min_distance:
                min_distance = distance
                closest_cluster = cluster_id
        
        cluster_points_single[closest_cluster].append(point)
        new_clusters.append(closest_cluster)
    
    # Convert lists to numpy arrays
    for cluster_id in cluster_points_single:
        cluster_points_single[cluster_id] = np.array(cluster_points_single[cluster_id], dtype=np.float32)

    return cluster_points_single, new_clusters

In [ ]:
trial = 2
dataset = "test"  # "train"  # "test"  # "single"  # "A4"  # "NACC_single"  

print(f"Trial {trial} ({dataset}):")
az_train, _, _, _, _, _, _, _, _, _ = process_data(trial, "train", impute_lb=True)
az, aRID, alb, aage, uz, uRID, ulb, uage, amyloid_dict, dataset_df = process_data(trial, dataset, impute_lb=True)

cluster_points, normal_branch, AD_branch, visualize_dict = initial_trajectory(az_train, branch_degree=3)

if dataset == "train":
    pseudotime_model = None
else:
    if pseudotime_model is None:
        raise ValueError("No pseudotime model from training data.")
    cluster_points, new_clusters = cluster_new_points(cluster_points, az)
    visualize_dict["X"] = az
    visualize_dict["new_labels"] = new_clusters

if trial == 2:
    # if dataset == "train":
        branch_points = [1]
        normal_branch = [2, 4, 5, 7, 8, 10, 11]
        AD_branch = [3, 6, 9, 12, 13, 14, 15, 16, 17]
    # elif dataset == "test":
    #     branch_points = [5]
        # normal_branch = [3, 4, 6, 8, 10]
        # AD_branch = [3, 5, 7, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19]

visualize_trajectory(
    visualize_dict["X"],
    visualize_dict["new_labels"],
    visualize_dict["cluster_centers"],
    visualize_dict["G"],
    visualize_dict["root"],
    branch_points,
    normal_branch,
    AD_branch,
    visualize_dict["pseudotime"]
)

normal_points = np.concatenate([cluster_points[i] for i in normal_branch if len(cluster_points[i]) > 0])
AD_points = np.concatenate([cluster_points[i] for i in AD_branch if len(cluster_points[i]) > 0])
common_points = np.array([point for point in az if not any(np.array_equal(point, p) for p in normal_points) and not any(np.array_equal(point, p) for p in AD_points)])
print("Normal_branch:", normal_points.shape, normal_branch)
print("AD_branch:    ", AD_points.shape, AD_branch)

apseudotime, normal_ptime, AD_ptime, pseudotime_model = cal_pseudotime(az, normal_points, AD_points, pseudotime_model)
normal_curve_points = np.vstack([common_points, normal_points])
AD_curve_points = np.vstack([common_points, AD_points])
normal_curve_RID = aRID[np.isin(az, normal_curve_points).all(axis=1)]
AD_curve_RID = aRID[np.isin(az, AD_curve_points).all(axis=1)]
normal_curve_age = aage[np.isin(az, normal_curve_points).all(axis=1)]
AD_curve_age = aage[np.isin(az, AD_curve_points).all(axis=1)]
# make normal_ptime and AD_ptime have the same length as apseudotime
ptime_normal = match_branch_ptime(az, normal_curve_points, apseudotime, normal_ptime)
ptime_AD = match_branch_ptime(az, AD_curve_points, apseudotime, AD_ptime)

if dataset not in ["single", "A4", "NACC_single"]:
    cal_violation(az, aRID, aage, apseudotime, thresholds=[0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])
cal_accuracy(uRID, uz, ulb, uage, common_points, normal_points, AD_points, amyloid_dict)
draw_subjs_w_label(az, uz, uRID, ulb, uage, amyloid_dict, 1)

# acluster records cluster of all points
acluster = np.zeros(az.shape[0], dtype=int)
abranch = np.zeros(az.shape[0], dtype=object)
for i in range(len(az)):
    for tmp_cluster in range(len(cluster_points)):
        if az[i] in cluster_points[tmp_cluster]:
            acluster[i] = tmp_cluster
            if tmp_cluster in normal_branch:
                abranch[i] = "normal_branch"
            elif tmp_cluster in AD_branch:
                abranch[i] = "AD_branch"
            else:
                abranch[i] = "common_branch"
            break

In [ ]:
# ---------------------
# Save results
# ---------------------
output_csv = f"./analysis/trial{trial}_{dataset}_pre.csv"
df_output = pd.DataFrame({
    "RID": aRID,
    "lb": alb,
    "age": aage,
    "cluster": acluster,
    "branch": abranch,
    "pseudotime": apseudotime,
    "ptime_normal": ptime_normal,
    "ptime_AD": ptime_AD,
    "embedding1": az[:, 0],
    "embedding2": az[:, 1],
})
# unpreadjusted data
# raw_dfs = [pd.read_csv(f"data/ADNI1GO234/filtered/multi_visit_data.csv"),
#            pd.read_csv(f"data/ADNI1GO234/filtered/single_visit_data.csv")]
# raw_df = pd.concat(raw_dfs, ignore_index=True)

raw_df = dataset_df
append_columns = raw_df.columns[2:5]
cognition_columns = raw_df.columns[6:-68]
features_columns = raw_df.columns[-68:]

trimmed_raw_df = raw_df[['RID', 'AGE'] + append_columns.tolist() + cognition_columns.tolist() + features_columns.tolist()]
trimmed_raw_df = trimmed_raw_df.rename(columns={'AGE': 'age'})
trimmed_raw_df['RID'] = trimmed_raw_df['RID'].astype(str)
trimmed_raw_df['age'] = trimmed_raw_df['age'].astype(float).round(2)

df_output = pd.merge(df_output, trimmed_raw_df, on=['RID', 'age'], how='left')

# save to CSV
df_output.to_csv(output_csv, index=False)